# Step 08B — Expert-added caries boxes: paired sensitivity evaluation

**Research objective:** Compare the **frozen YOLO11s model** against the original source-defined ground truth and against the **partially expert-corrected ground truth** (22 added boxes on 11 previously unlabeled images). **No training or model selection.**

**Important:** This does **not** rectify existing inaccurate boxes in other radiographs. Treat the corrected evaluation as exploratory; keep original source-GT metrics as primary, and do not pretend that the new labels were used during initial model development.

Upload **`Step_08B_CVAT_Approved_Inputs.zip`** when prompted. It contains the two CVAT exports and a pre-recorded Step 01B image/label SHA-256 manifest. Set Colab to **T4 GPU** before running. Do not expose images or expert patient-level data publicly without appropriate permissions.


In [ ]:
%pip -q install "ultralytics==8.4.139" numpy pandas matplotlib pyyaml pillow scipy


## 1. Mount Drive, check model identity, and upload approved correction bundle
The notebook does not modify the source dataset directories or weights. Any failure below should be investigated rather than bypassed.
**Upload fix:** If Colab has already renamed the archive to `Step_08B_CVAT_Approved_Inputs (3).zip`, the code finds and reuses it. If none is present, it accepts renamed uploads automatically.


In [ ]:
from google.colab import drive,files
drive.mount('/content/drive')
from pathlib import Path
import hashlib,zipfile,json,io,os,shutil,sys,platform,yaml,random,math
from datetime import datetime,timezone
from collections import Counter
import numpy as np,pandas as pd,torch,ultralytics
from ultralytics import YOLO
from ultralytics.utils.metrics import ap_per_class
from PIL import Image,ImageDraw,ImageFont
assert ultralytics.__version__=='8.4.139',ultralytics.__version__
assert torch.cuda.is_available(), 'Please select GPU: Runtime > Change runtime type > T4 GPU.'
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
WEIGHTS=ROOT/'final_model/YOLO11s_Final/final_60epochs.pt'
EXPECTED_MODEL='19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e'
ORIGINAL={'internal':ROOT/'processed_data/yolo_caries_baseline','external':ROOT/'external_validation/MOPG7_66'}
EXPECTED_COUNTS={'internal':(30,108),'external':(66,55)}
EXPECTED_ADDED={'internal':(3,7),'external':(8,15)}
EXPECTED_CORR_FILES={'internal':{'cate1_009.png','cate1_010.png','cate2_010.png'},'external':{'00001247.jpg','00001373.jpg','00001381.jpg','0000538.jpg','0000623.jpg','ibn159.jpg','ibn285.jpg','ibn59.jpg'}}
EXPECTED_ZIP_HASH={'internal':'27e8238dbaf2cb48fa6d75eb8eeeca9567ddd6fd0e77db83ed101d43c1bb86e9','external':'634478697e7f8e6842afe00db7ea6f13165e9149b811e2df3f30bc666ebfb347'}
def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
 return h.hexdigest()
assert sha256(WEIGHTS)==EXPECTED_MODEL,'STOP: Frozen checkpoint SHA-256 mismatch.'
print('Model verified:',EXPECTED_MODEL,'GPU:',torch.cuda.get_device_name(0))
# Robust Colab upload: reuse a ZIP already in /content (including '(2)' or '(3)' suffixes).
REQUIRED_MEMBERS={'Internal_3_CVAT_Annotations.zip','External_8_CVAT_Annotations.zip','Step01B_Original_96_SHA256_Manifest.csv'}
def is_approved_bundle(path):
 try:
  with zipfile.ZipFile(path) as archive:
   return REQUIRED_MEMBERS.issubset(set(archive.namelist())) and archive.testzip() is None
 except (zipfile.BadZipFile, OSError):
  return False

existing=sorted(Path('/content').glob('Step_08B_CVAT_Approved_Inputs*.zip'),
                key=lambda p:p.stat().st_mtime,reverse=True)
valid_existing=[p for p in existing if is_approved_bundle(p)]
if valid_existing:
 SOURCE_ARCHIVE=valid_existing[0]
 print('Reusing already uploaded ZIP:',SOURCE_ARCHIVE.name,'— no upload required.')
else:
 print('Select Step_08B_CVAT_Approved_Inputs.zip (Colab may append (2), (3), etc.):')
 uploaded=files.upload()
 valid_uploads=[(filename,contents) for filename,contents in uploaded.items()
                if Path(filename).name.startswith('Step_08B_CVAT_Approved_Inputs')
                and Path(filename).suffix.lower()=='.zip']
 assert len(valid_uploads)==1,('Expected exactly one approved-input ZIP:',list(uploaded))
 SOURCE_ARCHIVE=Path('/content/Step_08B_CVAT_Approved_Inputs.zip')
 SOURCE_ARCHIVE.write_bytes(valid_uploads[0][1])
 assert is_approved_bundle(SOURCE_ARCHIVE),'STOP: ZIP is unreadable or missing required CVAT files.'
 print('Accepted uploaded file:',valid_uploads[0][0])
print('PASS: Approved input ZIP located and verified:',SOURCE_ARCHIVE,'bytes:',SOURCE_ARCHIVE.stat().st_size)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_08B_Expert_Corrected_GT_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
INPUT_DIR=Path('/content/step08b_input');INPUT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as z:
 assert set(z.namelist())>= {'Internal_3_CVAT_Annotations.zip','External_8_CVAT_Annotations.zip','Step01B_Original_96_SHA256_Manifest.csv'}
 for name in ['Internal_3_CVAT_Annotations.zip','External_8_CVAT_Annotations.zip','Step01B_Original_96_SHA256_Manifest.csv']:
  (INPUT_DIR/name).write_bytes(z.read(name))
ORIGINAL_MANIFEST=pd.read_csv(INPUT_DIR/'Step01B_Original_96_SHA256_Manifest.csv')
assert len(ORIGINAL_MANIFEST)==96 and len(ORIGINAL_MANIFEST[['cohort','image']].drop_duplicates())==96
print('Output folder:',OUT)


## 2. Rebuild isolated corrected labels and perform quality checks
This uses a **new temporary test directory** under `/content`. It preserves all original image pixels and all non-target annotations unchanged. Labels for exactly the 11 approved images are replaced with their CVAT-exported boxes.

In [ ]:
STAGE=Path('/content/step08b_corrected_staging')
if STAGE.exists():shutil.rmtree(STAGE)
ADDED=[];image_audit=[]
for cohort,src in ORIGINAL.items():
 folder=STAGE/cohort
 (folder/'images/test').mkdir(parents=True)
 (folder/'labels/test').mkdir(parents=True)
 cvatfile=INPUT_DIR/('Internal_3_CVAT_Annotations.zip' if cohort=='internal' else 'External_8_CVAT_Annotations.zip')
 assert sha256(cvatfile)==EXPECTED_ZIP_HASH[cohort],'STOP: CVAT archive changed after approval.'
 with zipfile.ZipFile(cvatfile) as z:
  assert z.testzip() is None
  labnames={Path(n).stem:n for n in z.namelist() if n.startswith('labels/train/') and n.endswith('.txt')}
  assert set(labnames)=={Path(i).stem for i in EXPECTED_CORR_FILES[cohort]},'STOP: CVAT image list differs.'
  cvat_labels={stem:z.read(name) for stem,name in labnames.items()}
 images=sorted(p for p in (src/'images/test').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png','.bmp','.tiff','.tif'))
 assert len(images)==EXPECTED_COUNTS[cohort][0]
 total_original=0;total_corrected=0;n_added=0
 for p in images:
  orig_label=src/'labels/test'/f'{p.stem}.txt';assert orig_label.exists()
  record=ORIGINAL_MANIFEST[(ORIGINAL_MANIFEST.cohort==cohort)&(ORIGINAL_MANIFEST.image==p.name)]
  assert len(record)==1,p.name
  ref=record.iloc[0]
  assert sha256(p)==ref.image_sha256 and sha256(orig_label)==ref.label_sha256, ('STOP: source image or label has changed',cohort,p.name)
  original_rows=[line for line in orig_label.read_text().splitlines() if line.strip()]
  total_original+=len(original_rows)
  if p.name in EXPECTED_CORR_FILES[cohort]:
   assert len(original_rows)==0, ('STOP: CVAT update is not to a source-negative image',p.name)
   labbytes=cvat_labels[p.stem]
  else:labbytes=orig_label.read_bytes()
  # Validate normalized YOLO class 0 and exact five columns; require positive in-image extents
  corr_rows=[line for line in labbytes.decode().splitlines() if line.strip()]
  for line in corr_rows:
   v=line.split(); assert len(v)==5,(p.name,line)
   cls,x,y,w,h=map(float,v)
   assert cls==0 and all(np.isfinite([x,y,w,h])) and w>0 and h>0 and x-w/2>=-1e-7 and y-h/2>=-1e-7 and x+w/2<=1+1e-7 and y+h/2<=1+1e-7,(p.name,line)
  if p.name in EXPECTED_CORR_FILES[cohort]:
   assert len(corr_rows)>0
   for k,line in enumerate(corr_rows,1):ADDED.append(dict(cohort=cohort,image=p.name,box_index=k,yolo_line=line))
   n_added+=1
  shutil.copy2(p,folder/'images/test'/p.name)
  (folder/'labels/test'/f'{p.stem}.txt').write_bytes(labbytes)
  assert sha256(folder/'images/test'/p.name)==ref.image_sha256
  total_corrected+=len(corr_rows)
  image_audit.append(dict(cohort=cohort,image=p.name,GT_original=len(original_rows),GT_corrected=len(corr_rows),added=len(corr_rows)-len(original_rows),image_SHA256=ref.image_sha256,original_label_SHA256=ref.label_sha256,corrected_label_SHA256=sha256(folder/'labels/test'/f'{p.stem}.txt')))
 assert total_original==EXPECTED_COUNTS[cohort][1]
 assert n_added==EXPECTED_ADDED[cohort][0] and total_corrected-total_original==EXPECTED_ADDED[cohort][1]
 spec={'path':str(folder),'train':'images/test','val':'images/test','test':'images/test','names':{0:'caries'},'nc':1}
 (OUT/f'{cohort}_expert_added.yaml').write_text(yaml.safe_dump(spec,sort_keys=False))
record_df=pd.DataFrame(image_audit);record_df.to_csv(OUT/'approved_GT_change_manifest.csv',index=False)
pd.DataFrame(ADDED).to_csv(OUT/'new_expert_approved_boxes.csv',index=False)
print(record_df.groupby('cohort')[['GT_original','GT_corrected','added']].sum())
print('PASS: original 96 image SHA-256 and labels verified against Step 01B, 11 replacement files, 22 added boxes.')


## 3. Validator-aligned AP analysis, same frozen weights / settings
Revalidate the ORIGINAL and CORRECTED GT for each cohort; do not mix results from `predict()` and `val()`. Four evaluations at confidence 0.001 and four at confidence 0.25; all settings locked.

In [ ]:
IMGSZ=1024;NMS_IOU=0.70;MAX_DET=300;IOU_LEVELS=np.round(np.arange(.50,.951,.05),2)
def as_numpy(x):
 return x.detach().cpu().numpy().copy() if isinstance(x,torch.Tensor) else np.asarray(x).copy()
def capture_run(cohort,variant,conf):
 stats=[];state={'previous':0}
 def callback(v):
  s=v.metrics.stats;names=list(v.metrics.box.image_metrics.keys());n=len(s['tp'])
  assert len(names)==n
  for i in range(state['previous'],n):
   tp=as_numpy(s['tp'][i]).astype(bool).reshape(-1,10)
   scores=as_numpy(s['conf'][i]).astype(float).reshape(-1)
   target=as_numpy(s['target_cls'][i]).astype(int).reshape(-1)
   pc=as_numpy(s['pred_cls'][i]).astype(int).reshape(-1)
   assert len(scores)==len(tp)==len(pc) and np.all(pc==0) and np.all(target==0)
   stats.append(dict(image=names[i],tp=tp,scores=scores,pred_cls=pc,n_gt=len(target)))
  state['previous']=n
 model=YOLO(str(WEIGHTS));model.add_callback('on_val_batch_end',callback)
 if variant=='original':
  folder=ORIGINAL[cohort];s={'path':str(folder),'train':'images/test','val':'images/test','test':'images/test','names':{0:'caries'},'nc':1}
  yamlpath=OUT/f'{cohort}_original.yaml';yamlpath.write_text(yaml.safe_dump(s,sort_keys=False))
 else: yamlpath=OUT/f'{cohort}_expert_added.yaml'
 metrics=model.val(data=str(yamlpath),split='test',imgsz=IMGSZ,conf=conf,iou=NMS_IOU,max_det=MAX_DET,
                   device=0,half=False,workers=0,batch=4,save=False,save_txt=False,save_json=False,
                   plots=False,project=str(OUT/'validation_logs'),name=f'{cohort}_{variant}_{conf}',exist_ok=False,verbose=False)
 assert len(stats)==EXPECTED_COUNTS[cohort][0]
 assert set(x['image'] for x in stats)==set(record_df[record_df.cohort==cohort].image)
 expected=dict(zip(record_df[record_df.cohort==cohort].image,record_df[record_df.cohort==cohort]['GT_original' if variant=='original' else 'GT_corrected']))
 assert all(x['n_gt']==expected[x['image']] for x in stats),'STOP: labels seen by Ultralytics are not those audited.'
 return stats,dict(cohort=cohort,variant=variant,conf=conf,mAP50=float(metrics.box.map50),mAP50_95=float(metrics.box.map),model_precision=float(metrics.box.mp),model_recall=float(metrics.box.mr),n_predictions=sum(len(x['scores']) for x in stats))
def native_ap(items):
 tp=np.concatenate([a['tp'] for a in items]);scores=np.concatenate([a['scores'] for a in items]);pc=np.concatenate([a['pred_cls'] for a in items]);gt=np.zeros(sum(a['n_gt'] for a in items),dtype=int)
 if len(gt)==0:return np.full(10,np.nan)
 if len(scores)==0:return np.zeros(10)
 v=ap_per_class(tp,scores,pc,gt,plot=False)[5]
 assert v.shape==(1,10)
 return v[0]
STAT={};results=[]
for cohort in ['internal','external']:
 for variant in ['original','expert_added']:
  for conf in [0.001,0.25]:
   print('VALIDATING',cohort,variant,conf)
   st,r=capture_run(cohort,variant,conf);STAT[(cohort,variant,conf)]=st;results.append(r)
res=pd.DataFrame(results);res.to_csv(OUT/'official_validator_metrics.csv',index=False)
print(res)
# Strict historical baseline reproduction checks
ir=res[(res.cohort=='internal')&(res.variant=='original')&(res.conf==0.001)].iloc[0]
er=res[(res.cohort=='external')&(res.variant=='original')&(res.conf==0.001)].iloc[0]
assert abs(ir.mAP50-0.497526)<.0001 and abs(ir.mAP50_95-0.245775)<.0001,'STOP: original internal baseline is not reproduced.'
assert abs(er.mAP50)<1e-8 and abs(er.mAP50_95)<1e-8,'STOP: original external baseline mismatch.'
print('PASS: original AP reproduces Step 01B snapshot.')


## 4. Confirm AP parity and compute fixed-threshold/image-level counts
All AP must match the official Ultralytics validator statistics. The corrected GT can change the image-level labels, but **does not change any model prediction**.

In [ ]:
ap_parity=[]; per_image=[]; summaries=[];ap_by_iou=[]
def ratio(a,b):return float(a/b) if b else float('nan')
for cohort in ['internal','external']:
 for variant in ['original','expert_added']:
  x=res[(res.cohort==cohort)&(res.variant==variant)&(res.conf==0.001)].iloc[0]
  aps=native_ap(STAT[(cohort,variant,0.001)])
  d1=abs(aps[0]-x.mAP50);d2=abs(np.mean(aps)-x.mAP50_95)
  ap_parity.append(dict(cohort=cohort,variant=variant,AP50_validator=x.mAP50,AP50_reconstructed=aps[0],AP5095_validator=x.mAP50_95,AP5095_reconstructed=np.mean(aps),difference50=d1,difference5095=d2,pass_parity=d1<1e-5 and d2<1e-5))
  for i,a in enumerate(aps):ap_by_iou.append(dict(cohort=cohort,variant=variant,iou=IOU_LEVELS[i],AP=float(a)))
  items=STAT[(cohort,variant,0.25)]
  for t in items:
   tp=int(t['tp'][:,0].sum());nt=t['n_gt'];npr=len(t['scores'])
   per_image.append(dict(cohort=cohort,variant=variant,image=t['image'],nGT=nt,nPred=npr,TP=tp,FP=npr-tp,FN=nt-tp,image_GT_positive=int(nt>0),image_pred_positive=int(npr>0)))
  d=pd.DataFrame([v for v in per_image if v['cohort']==cohort and v['variant']==variant]);tp=int(d.TP.sum());fp=int(d.FP.sum());fn=int(d.FN.sum())
  a=int(((d.image_GT_positive==1)&(d.image_pred_positive==1)).sum());b=int(((d.image_GT_positive==0)&(d.image_pred_positive==1)).sum());c=int(((d.image_GT_positive==0)&(d.image_pred_positive==0)).sum());e=int(((d.image_GT_positive==1)&(d.image_pred_positive==0)).sum())
  summaries.append(dict(cohort=cohort,variant=variant,nImages=len(d),nGT=int(d.nGT.sum()),nPred=int(d.nPred.sum()),lesion_TP=tp,lesion_FP=fp,lesion_FN=fn,lesion_recall=ratio(tp,tp+fn),lesion_precision=ratio(tp,tp+fp),lesion_F1=ratio(2*tp,2*tp+fp+fn),image_TP=a,image_FP=b,image_TN=c,image_FN=e,image_sensitivity=ratio(a,a+e),image_specificity=ratio(c,c+b),image_accuracy=ratio(a+c,len(d)),image_F1=ratio(2*a,2*a+b+e)))
par=pd.DataFrame(ap_parity);assert par.pass_parity.all(),'STOP: AP reconstruction does not match official validator.'
par.to_csv(OUT/'mandatory_AP_parity.csv',index=False)
pd.DataFrame(ap_by_iou).to_csv(OUT/'AP_by_IoU_original_vs_corrected.csv',index=False)
per=pd.DataFrame(per_image);per.to_csv(OUT/'fixed_threshold_per_image.csv',index=False)
summ=pd.DataFrame(summaries);summ.to_csv(OUT/'fixed_threshold_and_image_level_summary.csv',index=False)
# Predictions should be identical even with GT replaced (same model and images).
for cohort in ['internal','external']:
 for conf in [.001,.25]:
  oa={x['image']:x['scores'] for x in STAT[(cohort,'original',conf)]}
  ca={x['image']:x['scores'] for x in STAT[(cohort,'expert_added',conf)]}
  assert all((len(oa[n])==len(ca[n]) and np.allclose(oa[n],ca[n],rtol=1e-6,atol=2e-6)) for n in oa),'STOP: prediction confidences changed between original and corrected GT.'
# Historical Step 01B fixed-threshold totals provide an additional independent gate.
for cohort, expected_tp,expected_fp,expected_fn in [('internal',9,6,99),('external',0,6,55)]:
 orig=summ[(summ.cohort==cohort)&(summ.variant=='original')].iloc[0]
 assert (orig.lesion_TP,orig.lesion_FP,orig.lesion_FN)==(expected_tp,expected_fp,expected_fn), ('STOP: original operating point is not reproduced',cohort,orig.to_dict())
print('PASS: AP parity, Step 01B operating-point totals, and unchanged prediction scores in every image.');display(summ)


## 5. Paired radiograph bootstrap, 2,000 replicates
For each cohort, resample the *same* radiograph indices for original and corrected GT. Report 95% percentile intervals for AP and **paired differences**. As in Step 01B, no models are retrained.

In [ ]:
RNGSEED=42;B=2000; boot_rows=[]
for cohort in ['internal','external']:
 names=sorted(EXPECTED_CORR_FILES[cohort]|set(record_df[record_df.cohort==cohort].image))
 ap0={x['image']:x for x in STAT[(cohort,'original',0.001)]};ap1={x['image']:x for x in STAT[(cohort,'expert_added',0.001)]}
 fx0=per[(per.cohort==cohort)&(per.variant=='original')].set_index('image');fx1=per[(per.cohort==cohort)&(per.variant=='expert_added')].set_index('image')
 rng=np.random.default_rng(RNGSEED)
 for i in range(B):
  chosen=[names[k] for k in rng.integers(0,len(names),len(names))]
  p0=native_ap([ap0[n] for n in chosen]);p1=native_ap([ap1[n] for n in chosen]);
  row=dict(cohort=cohort,replicate=i+1,AP50_source=float(p0[0]),AP50_corrected=float(p1[0]),AP50_difference=float(p1[0]-p0[0]),AP5095_source=float(p0.mean()),AP5095_corrected=float(p1.mean()),AP5095_difference=float(p1.mean()-p0.mean()))
  for variant,df in [('source',fx0),('corrected',fx1)]:
   d=df.loc[chosen]
   tp=int(d.TP.sum());fp=int(d.FP.sum());fn=int(d.FN.sum())
   a=int(((d.image_GT_positive==1)&(d.image_pred_positive==1)).sum());b=int(((d.image_GT_positive==0)&(d.image_pred_positive==1)).sum());c=int(((d.image_GT_positive==0)&(d.image_pred_positive==0)).sum());e=int(((d.image_GT_positive==1)&(d.image_pred_positive==0)).sum())
   row[f'lesion_recall_{variant}']=ratio(tp,tp+fn)
   row[f'image_accuracy_{variant}']=ratio(a+c,len(d))
  row['image_accuracy_difference']=row['image_accuracy_corrected']-row['image_accuracy_source']
  boot_rows.append(row)
 print('Bootstrap complete',cohort)
boot=pd.DataFrame(boot_rows);boot.to_csv(OUT/'paired_bootstrap_2000.csv',index=False)
ci=[]
for cohort,d in boot.groupby('cohort'):
 for field in [x for x in d.columns if x not in ('cohort','replicate')]:
  x=d[field].dropna().to_numpy(float)
  ci.append(dict(cohort=cohort,metric=field,CI_2_5=float(np.quantile(x,.025)) if len(x) else float('nan'),CI_97_5=float(np.quantile(x,.975)) if len(x) else float('nan'),n_valid=len(x)))
pd.DataFrame(ci).to_csv(OUT/'paired_bootstrap_95CI.csv',index=False)
print('Paired radiograph bootstrap finished.')


## 6. Create expert-approval overlays for the exact 11 added-positive images
These overlays are evidence of the exported coordinates on the original images. Keep the expert approval email / log in your research records and do not publish clinical labels without the applicable dataset permissions.

In [ ]:
OVER=OUT/'new_approved_GT_overlays';OVER.mkdir()
for rec in record_df.itertuples(index=False):
 if rec.added<=0:continue
 imgpath=STAGE/rec.cohort/'images/test'/rec.image
 with Image.open(imgpath) as im:
  im=im.convert('RGB')
  d=ImageDraw.Draw(im);W,H=im.size
  label=(STAGE/rec.cohort/'labels/test'/f'{Path(rec.image).stem}.txt').read_text().splitlines()
  for i,line in enumerate(label,1):
   _,cx,cy,w,h=map(float,line.split())
   x1=(cx-w/2)*W;y1=(cy-h/2)*H;x2=(cx+w/2)*W;y2=(cy+h/2)*H
   d.rectangle([x1,y1,x2,y2],outline=(255,0,0),width=max(2,round(W/700)))
   d.text((x1,max(0,y1-16)),f'GT {i}',fill=(255,255,0),stroke_width=1,stroke_fill=(0,0,0))
  im.save(OVER/f'{rec.cohort}__{Path(rec.image).stem}.png')
print('Generated',len(list(OVER.glob('*.png'))),'expert-corrected overlays.')


## 7. Build final evidence archive
The archive includes the **paired source-vs-corrected numerical analysis**, the 11 image annotation overlays, model/input manifest, and strict AP parity results. Do not submit the work to the journal until these values are reviewed and incorporated consistently into manuscript tables and response letters.

In [ ]:
summary={
 'analysis':'Exploratory, partially expert-corrected GT: 11 previously source-negative images, 22 new caries boxes.',
 'source_images_unmodified':True,'source_labels_unmodified':True,'model_sha256':EXPECTED_MODEL,
 'ultralytics_version':ultralytics.__version__,'imgsz':IMGSZ,'conf_AP':0.001,'conf_operating':0.25,
 'NMS_iou':NMS_IOU,'max_det':MAX_DET,'bootstrap_replicates':B,
 'caution':'Other expert-flagged inaccurate legacy boxes are not corrected; this is not full reference-standard harmonization.'}
(OUT/'README_ANALYSIS_LIMITATIONS.json').write_text(json.dumps(summary,indent=2))
archive=OUT/'Step_08B_Expert_Corrected_GT_Sensitivity_Results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in OUT.rglob('*'):
  if not p.is_file() or p==archive:continue
  if 'validation_logs' in p.parts:continue
  z.write(p,p.relative_to(OUT))
print('SUCCESS: Stage images and 96 labels unchanged in source directories. Output:',archive)
print('ZIP size:',archive.stat().st_size,'bytes')
